In [1]:
import requests
import time
from mal_scraper import get_user_anime_list
import logging
import pandas as pd
import numpy as np
import os

In [2]:
logging.getLogger('jikanpy').setLevel(logging.CRITICAL)
logging.getLogger().setLevel(logging.CRITICAL)

In [3]:
delay = 1
maxUserSize = 2000

In [4]:
def getRandomUser():
    time.sleep(delay)
    url = "https://api.jikan.moe/v4/random/users"
    try:
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            return data['data']
        elif response.status_code == 429:
            print("Rate limited! Waiting before retry...")
            time.sleep(0.1)
            return getRandomUser()
        else:
            print(f"Error: {response.status_code}")
            return None
            
    except Exception as e:
        print(f"An error occurred: {e}")
        return None

In [5]:
username = "Exarfate"
print(username)

Exarfate


In [17]:
def getReviewsFromUser(username):
    anime_list = get_user_anime_list(username)
    time.sleep(delay)
    if anime_list is None: return []
    anime_list = [anime for anime in anime_list if anime['score'] > 0]
    return anime_list

In [7]:
anime_list = getReviewsFromUser(username)
print(anime_list)
print(len(anime_list))

[{'name': 'Code Geass: Hangyaku no Lelouch', 'id_ref': 1575, 'consumption_status': <ConsumptionStatus.completed: 'COMPLETED'>, 'is_rewatch': False, 'score': 10, 'progress': 25, 'start_date': None, 'finished_date': None}, {'name': 'Code Geass: Hangyaku no Lelouch R2', 'id_ref': 2904, 'consumption_status': <ConsumptionStatus.completed: 'COMPLETED'>, 'is_rewatch': False, 'score': 10, 'progress': 25, 'start_date': None, 'finished_date': None}]
2


In [8]:
url = "https://api.jikan.moe/v4/top/anime"
while True:
    response = requests.get(url)
    
    time.sleep(delay)
    
    data = response.json()
    if not 'status' in data or data['status'] != 500:
        break


In [9]:
url = "https://api.jikan.moe/v4/top/anime"
popular_anime_ids = []
for page in range(1, 20):
    params = {
        "filter": "bypopularity",
        "page": page,
    }
    response = requests.get(url, params=params)
    time.sleep(delay)
    data = response.json()
    # print(data)
    animes = data['data']
    popular_anime_ids += [anime['mal_id'] for anime in animes]

print(popular_anime_ids)
print(len(popular_anime_ids))

[16498, 1535, 5114, 30276, 38000, 31964, 11757, 11061, 20, 22319, 40748, 32281, 25777, 9253, 1735, 33486, 21, 35760, 28851, 38524, 19815, 1575, 31240, 23273, 36456, 4224, 32182, 20507, 31043, 40028, 22199, 24833, 23755, 269, 6547, 20583, 37779, 30831, 10620, 21881, 1, 9919, 22535, 30, 199, 33352, 44511, 37450, 38691, 2904, 37999, 28223, 38408, 34572, 27899, 50265, 34134, 14719, 18679, 6702, 37521, 35849, 11111, 40456, 3588, 2001, 35790, 28999, 29803, 13601, 28171, 47778, 37510, 9989, 15809, 32937, 28121, 37430, 42897, 34933, 226, 8074, 10087, 39535, 30654, 14813, 28891, 48583, 121, 30503, 31478, 34599, 38671, 11617, 2167, 40591, 5081, 6746, 431, 12189, 35507, 14741, 26243, 52991, 9756, 42249, 51009, 164, 37520, 20899, 41587, 205, 7054, 19, 33206, 32935, 48736, 813, 13759, 6880, 11771, 36511, 4181, 39587, 23283, 10793, 35120, 4898, 31933, 48561, 33255, 26055, 18897, 18153, 37349, 853, 34577, 32282, 40852, 11741, 17265, 22297, 37991, 14227, 523, 23847, 918, 20785, 36474, 52299, 30015, 22

In [10]:
def get_anime_reviews_jikan(anime_id, page):
    """
    Get reviews for an anime using Jikan API
    """
    url = f"https://api.jikan.moe/v4/anime/{anime_id}/reviews?page={page}"
    #params = {"page": page,}
    print(url)
    
    response = requests.get(url)
    time.sleep(delay)
    
    if response.status_code == 200:
        return response.json()
    else:
        return None

In [11]:
number = 1
t0 = time.time()
distinctUsers = set()
start = 0

In [12]:
from pathlib import Path

while True:
    file_path = Path(f"aaa_{start}")
    if not file_path.exists():
        break
        
    with open(file_path, 'r') as f:
        for line in f:
            user = line.strip()
            if user:
                distinctUsers.add(user)
    
    start += 1

print(f"Found {len(distinctUsers)} unique users.")

Found 0 unique users.


In [13]:
for ind in range(start, len(popular_anime_ids)):
    animeId = popular_anime_ids[ind]
    print(animeId)
    page = 1
    while True:
        print()
    # for page in range(1, 10):        
        data = get_anime_reviews_jikan(animeId, page)
        # print("AAA", page, data)
        print(data)

        if not data or ('status' in data and data['status'] == 500):
            page += 1
            continue

        number_of_failed_attempts = 0
            
        for elem in data['data']:
            for i, review in enumerate(data['data'], 1):
                # print(f"\nReview {i}:")
                # print(f"User: {review['user']['username']}")
                # print(f"Score: {review['score']}")
                distinctUsers.add(review['user']['username'])

        if not data['pagination']['has_next_page']:
            print(f"{number} ended at: {page} with total of {len(distinctUsers)} users after {time.time() - t0} seconds")
            break
        page += 1
    number += 1
    if len(distinctUsers) >= maxUserSize:
        print(f"Ending with {len(distinctUsers)}")
        break

start = ind

16498

https://api.jikan.moe/v4/anime/16498/reviews?page=1
{'pagination': {'last_visible_page': 1, 'has_next_page': True}, 'data': [{'mal_id': 163251, 'url': 'https://myanimelist.net/reviews.php?id=163251', 'type': 'anime', 'reactions': {'overall': 1794, 'nice': 1733, 'love_it': 32, 'funny': 8, 'confusing': 4, 'informative': 2, 'well_written': 15, 'creative': 0}, 'date': '2014-10-02T01:53:00+00:00', 'review': 'Oh dear Shingeki no Kyojin, where do I even begin. If you\'ve talked with your friends about anime, then the couple anime that everyone talks about are Naruto, Bleach, One Piece, Dragon Ball, and... Shingeki no Kyojin. What\'s the difference between Shingeki and the rest? Shingeki only has 25 episodes so far yet it\'s on par in popularity with the other super long, Americanized anime. Why is it popular? Well that\'s simply because it\'s stunningly amazing. Those people that call Shingeki no Kyojin "overrated" may not have the same taste as me, and that\'s perfectly fine, but in m

In [14]:
print(len(distinctUsers))

2147


In [15]:
usernames = list(distinctUsers)

In [16]:
rows = []
rowsAtLeast5 = []
usernamesAtLeast5 = []
for username in usernames:
    animeList = getReviewsFromUser(username)
    shortedAnimeList = [{
            'id_ref': entry['id_ref'],
            'name': entry['name'],
            'score': entry['score'],
        } 
        for entry in animeList]
    # print(animeList)
    # print(shortedAnimeList)
    # print(len(animeList))
    rows.append(shortedAnimeList)
    if len(shortedAnimeList) > 4:
        rowsAtLeast5.append(shortedAnimeList)
        usernamesAtLeast5.append(username)

c:\Users\mateu\AppData\Local\Python\pythoncore-3.13-64\Lib\site-packages\mal_scraper\users.py:341: SyntaxWarning: invalid escape sequence '\/'
  "video_url":"\/anime\/32998\/91_Days\/video",


ConnectionError: ('Connection aborted.', ConnectionResetError(10054, 'Istniejące połączenie zostało gwałtownie zamknięte przez zdalnego hosta', None, 10054, None))

In [ ]:
with open("usernames.txt", "w") as f:
    for item in distinctUsers:
        f.write(f"{item}\n")

In [ ]:
df = pd.DataFrame([
    {f"{item['id_ref']}_{item['name']}": item['score'] for item in userAnime}
    for userAnime in rows
], index=usernames)

df = df.fillna(0)

df

,32998_91 Days,11759_Accel World,31580_Ajin,6547_Angel Beats!,9989_Ano Hi Mita Hana no Namae wo Bokutachi wa Mada Shiranai.,24833_Ansatsu Kyoushitsu,30654_Ansatsu Kyoushitsu 2nd Season,34443_Baki,288_Bakuten Shoot Beyblade,37498_Black Fox,...,10507_Inazuma Eleven Go,9032_Inazuma Eleven: Saikyou Gundan Ogre Shuurai,50325_Kaguya-sama wa Kokurasetai: Ultra Romantic Teaser PV - Ishigami Yuu wa Kataritai,39651_Mob Psycho 100: Dai Ikkai Rei toka Soudansho Ian Ryokou - Kokoro Mitasu Iyashi no Tabi,36616_Mob Psycho 100: Reigen - Shirarezaru Kiseki no Reinouryokusha,57864_Monogatari Series: Off & Monster Season,40489_Sword Art Online: Alicization - War of Underworld Reflection,36539_Tsuki ga Kirei Special,13261_Inazuma Eleven Go: Chrono Stone,33733_Inazuma Eleven: Ares no Tenbin
ItIsIDio,6.0,6.0,8.0,5.0,2.0,8.0,8.0,5.0,4.0,4.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Rhapsody-,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,5.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
JonerBaloner,0.0,0.0,0.0,8.0,9.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
CHZabuza,6.0,5.0,0.0,0.0,0.0,7.0,7.0,9.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Butterflight,0.0,0.0,0.0,9.0,9.0,7.0,8.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
vasolina,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Shirogari,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Tyrannicswine117,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
PoltergeistChan,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
df2 = pd.DataFrame([
    {f"{item['id_ref']}_{item['name']}": item['score'] for item in userAnime}
    for userAnime in rowsAtLeast5
], index=usernamesAtLeast5)

df2 = df2.fillna(0)

df2

,32998_91 Days,11759_Accel World,31580_Ajin,6547_Angel Beats!,9989_Ano Hi Mita Hana no Namae wo Bokutachi wa Mada Shiranai.,24833_Ansatsu Kyoushitsu,30654_Ansatsu Kyoushitsu 2nd Season,34443_Baki,288_Bakuten Shoot Beyblade,37498_Black Fox,...,10507_Inazuma Eleven Go,9032_Inazuma Eleven: Saikyou Gundan Ogre Shuurai,50325_Kaguya-sama wa Kokurasetai: Ultra Romantic Teaser PV - Ishigami Yuu wa Kataritai,39651_Mob Psycho 100: Dai Ikkai Rei toka Soudansho Ian Ryokou - Kokoro Mitasu Iyashi no Tabi,36616_Mob Psycho 100: Reigen - Shirarezaru Kiseki no Reinouryokusha,57864_Monogatari Series: Off & Monster Season,40489_Sword Art Online: Alicization - War of Underworld Reflection,36539_Tsuki ga Kirei Special,13261_Inazuma Eleven Go: Chrono Stone,33733_Inazuma Eleven: Ares no Tenbin
ItIsIDio,6.0,6.0,8.0,5.0,2.0,8.0,8.0,5.0,4.0,4.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Rhapsody-,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,5.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
JonerBaloner,0.0,0.0,0.0,8.0,9.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
CHZabuza,6.0,5.0,0.0,0.0,0.0,7.0,7.0,9.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Butterflight,0.0,0.0,0.0,9.0,9.0,7.0,8.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
PyraXadon,0.0,0.0,1.0,6.0,6.0,8.0,5.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Honksea,7.0,0.0,0.0,10.0,8.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Fueeru,0.0,5.0,6.0,8.0,9.0,8.0,10.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
GregRocks,0.0,0.0,0.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
df.to_csv('anime_df.csv')  # index=False removes row numbers

In [ ]:
df_ind = 1
while True:
    filename = f"anime_df_{df_ind}.csv"
    
    if not os.path.exists(filename):
        break
            
    df_ind += 1

In [ ]:
df.to_csv(f"anime_df_{df_ind}.csv")

In [ ]:
print(len(usernames))

678


In [ ]:
print(start)